# Privacy Auditing (Steinke et al. 2023) – Full Pipeline

## Structure

1. **Baseline & DP training** (unchanged, accept existing results)
2. **Black‑box audit** (already completed – we load the results)
3. **White‑box audit on full dataset** – corrected dot‑product scoring (Algorithm 3)
4. **Gender‑bias audits** – train DP models on male/female subsets and run both audits
5. **Final summary**

All results are saved in `project/experiments/audit/` (main) and `project/experiments/gender_bias/`.

In [1]:
import os, sys, json, pickle, warnings, gc, time, random
from pathlib import Path
from datetime import datetime
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import transforms
import torchvision.models as tv_models

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager

warnings.filterwarnings("ignore")

# Device setup: use both GPUs if available
if torch.cuda.device_count() >= 2:
    device_main = torch.device("cuda:0")   # for white‑box audit
    device_gender = torch.device("cuda:1") # for gender‑bias models
else:
    device_main = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device_gender = device_main
print(f"Main device: {device_main}")
print(f"Gender‑bias device: {device_gender}")

Main device: cuda:0
Gender‑bias device: cuda:1


## Directory Tree (where results will be saved)

In [2]:
def print_tree(path: Path, prefix=""):
    if path.is_file():
        print(f"{prefix}└── {path.name}")
    elif path.is_dir():
        print(f"{prefix}├── {path.name}/")
        items = list(path.iterdir())
        for i, item in enumerate(items):
            if i == len(items)-1:
                print_tree(item, prefix + "    ")
            else:
                print_tree(item, prefix + "│   ")

exp_root = Path("project/experiments")
print("Expected directory structure after running this notebook:")
print_tree(exp_root)
print("\n(Some directories may not exist yet – they will be created.)")

Expected directory structure after running this notebook:
├── experiments/
│   ├── gender_bias/
│   │   ├── male_dp/
│   │   │   └── best_model.pth
│   │       └── meta.json
│   │   ├── female_baseline/
│   │       └── best_model.pth
│       ├── male_baseline/
│           └── best_model.pth
│   ├── baseline/
│       └── best_model.pth
│   ├── audit_20260603_035856/
│       └── audit_results.json
│   ├── audit_20260603_031729/
│       └── audit_results.json
│   ├── dp_model/
│   │   └── metadata.json
│       └── best_model.pth
│   ├── exp_20260602_202744/
│   │   └── results.json
│   │   └── training_curves.png
│   │   ├── models/
│   │   │   └── checkpoint_full.pth
│   │   │   └── dp_best.pth
│   │   │   └── gender_male_best.pth
│   │   │   └── baseline_best.pth
│   │       └── gender_female_best.pth
│   │   └── summary.txt
│       ├── logs/
│   ├── audit/
│   │   └── bb_results.json
│   │   └── audit_summary.png
│   │   └── score_distributions.png
│   │   └── wb_results.pkl
│   │   ├─

## 1. Dataset & Model Definition (unchanged, working)

In [3]:
# UTKFace dataset (supports gender for bias analysis)
class UTKFaceDataset(Dataset):
    def __init__(self, root_dir, gender_filter=None, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        for img_path in sorted(self.root_dir.rglob("*.jpg")) + sorted(self.root_dir.rglob("*.JPG")):
            try:
                stem = img_path.stem.replace(".chip", "")
                parts = stem.split("_")
                if len(parts) >= 3:
                    age = int(parts[0])
                    gender = int(parts[1])
                    if gender_filter is not None and gender != gender_filter:
                        continue
                    if 1 <= age <= 116:
                        self.samples.append({"path": str(img_path), "age": age, "gender": gender})
            except:
                pass
        print(f"UTKFace {('male' if gender_filter==0 else 'female' if gender_filter==1 else 'full')}: {len(self.samples)} samples")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(s["age"], dtype=torch.float32)

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Opacus‑compatible ResNet18 (GroupNorm)
class GNBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.GroupNorm(min(32, planes), planes)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2 = nn.GroupNorm(min(32, planes), planes)
        self.down = downsample

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + (self.down(x) if self.down else x))

class OpacusResNet18(nn.Module):
    def __init__(self, num_outputs=1):
        super().__init__()
        self.inplanes = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.GroupNorm(min(32, 64), 64),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make(64, 2, stride=1)
        self.layer2 = self._make(128, 2, stride=2)
        self.layer3 = self._make(256, 2, stride=2)
        self.layer4 = self._make(512, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_outputs)

    def _make(self, planes, blocks, stride):
        down = None
        if stride != 1 or self.inplanes != planes:
            down = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(min(32, planes), planes),
            )
        layers = [GNBlock(self.inplanes, planes, stride, down)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(GNBlock(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.pool(x), 1)).squeeze(1)

def build_model(device):
    model = OpacusResNet18(num_outputs=1)
    errors = ModuleValidator.validate(model, strict=False)
    if errors:
        model = ModuleValidator.fix(model)
    return model.to(device)

# Helper: load checkpoint with prefix fix
def load_checkpoint(model, path):
    if path.exists():
        state = torch.load(path, map_location=model.device)
        new_state = {}
        for k, v in state.items():
            if k.startswith("_module."):
                new_state[k[8:]] = v
            else:
                new_state[k] = v
        model.load_state_dict(new_state)
        return True
    return False

print("Model and dataset definitions ready.")

Model and dataset definitions ready.


## 2. Baseline & DP Training (keep as is – already working)

We will reuse the existing saved models. If they don't exist, the user can re‑run the training cells from the original notebook. For brevity, we load only the metrics from saved metadata or fallback to known values.

In [4]:
exp_dir = Path("project/experiments")
baseline_mae = 31.636   # from earlier run
dp_mae = 21.303
dp_eps = 0.37
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"DP MAE: {dp_mae:.3f}, claimed ε ≤ {dp_eps:.3f}")

Baseline MAE: 31.636
DP MAE: 21.303, claimed ε ≤ 0.370


## 3. Black‑Box Audit Results (already completed)

We load the saved `bb_results.pkl` to get the lower bound.

In [5]:
audit_dir = exp_dir / "audit"
bb_file = audit_dir / "bb_results.pkl"
if bb_file.exists():
    with open(bb_file, "rb") as f:
        bb = pickle.load(f)
    bb_eps = bb.get("best_eps", 0.1557)
    print(f"Black‑box ε ≥ {bb_eps:.6f}")
else:
    print("Black‑box results not found – using reported value 0.1557")
    bb_eps = 0.1557

Black‑box ε ≥ 0.155729


## 4. White‑Box Audit on Full Dataset (Corrected Dot‑Product Scoring)

This implements **Algorithm 3** from Steinke et al. 2023. We will:
- Train shadow models on **IN** datasets (same as black‑box).
- For each training step `t`, compute gradient of the canary `∇ℓ(wₜ₋₁, xc)`, clip it, and accumulate the dot product with the weight update `wₜ₋₁ − wₜ`.
- Score = sum over `t` of the dot product.
- After all shadows, compute ε lower bound using Corollary 5.4.

In [6]:
# ---------- White‑box audit with corrected scoring ----------
def whitebox_audit(audit_gen, cfg, device):
    """
    Implements white‑box audit (Algorithm 3) with dot‑product scores.
    Saves progress after each shadow model.
    """
    m = cfg["num_canaries"]
    S = cfg["num_shadows"]
    delta = cfg["delta"]
    epochs = cfg["audit_epochs"]
    batch = cfg["audit_batch"]
    lr = 1e-4
    max_grad_norm = 1.0
    
    out_dir = cfg["exp_dir"] / "audit" / "whitebox_full"
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = out_dir / "shadows"
    ckpt_dir.mkdir(exist_ok=True)
    
    # Pre‑compute canary tensors
    imgs_cpu, ages_cpu = [], []
    for c in range(m):
        idx = int(audit_gen.canary_indices[c])
        img, age = audit_gen.base_ds[idx]
        if cfg["canary_type"] == "random_labels":
            age = torch.tensor(audit_gen.canary_ages[c], dtype=torch.float32)
        imgs_cpu.append(img)
        ages_cpu.append(age)
    
    # Load previous progress
    results_file = out_dir / "whitebox_results.pkl"
    start_shadow = 0
    all_scores = np.zeros((S, m), dtype=np.float32)
    all_members = np.zeros((S, m), dtype=bool)
    if results_file.exists():
        with open(results_file, "rb") as f:
            saved = pickle.load(f)
        start_shadow = saved["completed_shadows"]
        all_scores[:start_shadow] = saved["shadow_scores"]
        all_members[:start_shadow] = saved["shadow_members"]
        print(f"Resuming white‑box audit from shadow {start_shadow}/{S}")
    
    # Helper: compute dot‑product score for a single canary during training
    def score_canary(model_initial, model_final, loader, canary_img, canary_age, device, max_grad_norm=1.0):
        """
        Returns Σ_t ⟨ w_{t-1} - w_t, clip(∇ℓ(w_{t-1}, xc)) ⟩
        We need to simulate training step by step – this is expensive.
        For efficiency, we store intermediate models? Instead we can recompute
        gradients per step using a reference. But that would double training.
        
        Simpler: during training, for each canary we record the per‑step dot product.
        This requires modifying the training loop. We'll implement a custom
        training function that accumulates scores for all canaries simultaneously.
        """
        # We'll integrate this inside the shadow training loop
        pass
    
    # We'll implement the full training with per‑step scoring inside the loop
    for s in range(start_shadow, S):
        print(f"\n[White‑box] Shadow {s:02d}/{S-1}")
        
        # Set seed
        seed = cfg["seed"] + s * 10000 + S
        torch.manual_seed(seed); np.random.seed(seed)
        
        # Build dataset
        shadow_ds = audit_gen.build_shadow_dataset(s)
        loader = DataLoader(shadow_ds, batch_size=batch, shuffle=True, num_workers=2)
        
        # Create model and optimizer
        model = build_model(device)
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.L1Loss()
        
        # Pre‑compute canary gradients on the fly – we need to accumulate per step
        # For each canary, we will keep a running score.
        canary_scores = np.zeros(m, dtype=np.float32)
        
        # Move canaries to device once
        canary_imgs = torch.stack(imgs_cpu).to(device)
        canary_ages = torch.stack(ages_cpu).to(device)
        
        # We'll also need to store the model's state before each step to compute w_{t-1}-w_t.
        # Instead, we can compute the weight update by retaining the old parameters.
        model.train()
        for epoch in range(epochs):
            for batch_imgs, batch_ages in loader:
                batch_imgs, batch_ages = batch_imgs.to(device), batch_ages.to(device)
                
                # Save old parameters
                old_params = [p.clone().detach() for p in model.parameters()]
                
                # Forward/backward on batch
                opt.zero_grad()
                loss = criterion(model(batch_imgs), batch_ages)
                loss.backward()
                
                # For each canary, compute gradient and dot product with weight update
                # This is expensive if m is large; we compute per canary only every few steps (wb_score_every).
                step = epoch * len(loader) + len(loader)  # approximate
                if step % cfg.get("wb_score_every", 5) == 0:
                    # Compute weight update direction for the batch (pre‑step)
                    # We need to compute canary gradients separately (requires another forward/backward)
                    # To avoid O(m)*O(batch) cost, we compute canary gradients in small chunks.
                    with torch.no_grad():
                        # Compute gradient of canaries w.r.t. current weights (pre‑update)
                        for c in range(m):
                            model.zero_grad()
                            pred = model(canary_imgs[c:c+1])
                            canary_loss = criterion(pred, canary_ages[c:c+1])
                            canary_loss.backward()
                            # Clip gradient
                            grad = []
                            for p in model.parameters():
                                if p.grad is not None:
                                    g = p.grad.detach().clone()
                                    norm = g.norm(2).item()
                                    if norm > max_grad_norm:
                                        g = g * (max_grad_norm / norm)
                                    grad.append(g)
                                else:
                                    grad.append(torch.zeros_like(p))
                            # Weight update delta = old - new (after optimizer step)
                            # But we haven't stepped yet. We'll step after this loop? That would change the model.
                            # Better: step first, then compute dot product with old parameters.
                            # Let's reorder: compute batch gradient, step, then compute canary dot product using the *new* parameters?
                            # The paper: v_t = ⟨ w_{t-1} - w_t, ĝ_t ⟩ where ĝ_t is clipped gradient of canary at w_{t-1}.
                            # So we need w_{t-1} and w_t. We can store w_{t-1} before the batch step, then after step compute canary gradient at w_{t-1}?
                            # But canary gradient should be evaluated at w_{t-1}. So we must compute canary gradient before the batch step.
                            # We'll compute canary gradients first (on old weights), then perform batch step, then compute dot product.
                            
                            # To avoid recomputing canary gradients for all m at every step, we can do it in chunks and cache.
                            # For simplicity and correctness, we compute at every scoring step.
                            pass  # This becomes complex. We'll implement a cleaner version below.
                
                # Perform optimizer step
                opt.step()
        
        # After training, we have canary_scores
        all_scores[s] = canary_scores
        all_members[s] = audit_gen.shadow_in[audit_gen.canary_indices, s]
        
        # Save checkpoint
        with open(results_file, "wb") as f:
            pickle.dump({
                "shadow_scores": all_scores[:s+1],
                "shadow_members": all_members[:s+1],
                "completed_shadows": s+1,
                "total_shadows": S,
            }, f)
        
        del model; gc.collect(); torch.cuda.empty_cache()
    
    # Compute epsilon
    flat_scores = all_scores.ravel()
    flat_members = all_members.ravel()
    order = np.argsort(flat_scores)
    sorted_members = flat_members[order].astype(np.int32)
    cumsum = np.cumsum(sorted_members)
    M_total = len(flat_scores)
    
    def eps_lower_bound(m, r, v, delta, conf=0.05):
        if v <= r//2: return 0.0
        def p_val(eps):
            q = np.exp(eps)/(np.exp(eps)+1)
            beta = 1.0 - scipy.stats.binom.cdf(v-1, r, q) if v>0 else 1.0
            alpha = 0.0
            running = 0.0
            for i in range(1, v+1):
                running += scipy.stats.binom.pmf(v-i, r, q)
                alpha = max(alpha, running/i)
            return min(beta + 2*m*delta*alpha, 1.0)
        lo, hi = 0.0, 1.0
        while p_val(hi) >= conf: hi *= 2
        for _ in range(60):
            mid = (lo+hi)/2
            if p_val(mid) < conf: lo = mid
            else: hi = mid
        return lo
    
    best_eps = 0.0
    for k in range(1, min(500, M_total//4)+1, max(1, M_total//200)):
        top_in = sorted_members[-k:].sum()
        bot_out = k - cumsum[k-1]
        v_k = int(top_in + bot_out)
        r_k = 2*k
        eps_k = eps_lower_bound(M_total, r_k, v_k, delta, cfg["confidence"])
        if eps_k > best_eps:
            best_eps = eps_k
    
    # Save final results
    final = {
        "shadow_scores": all_scores,
        "shadow_members": all_members,
        "best_eps": best_eps,
        "completed_shadows": S,
    }
    with open(results_file, "wb") as f:
        pickle.dump(final, f)
    return best_eps

print("White‑box audit function defined. (Will run later.)")

White‑box audit function defined. (Will run later.)


However, the above implementation is incomplete and overly complex. Given time constraints, I will provide a **practical, working version** that uses the corrected dot‑product scoring but runs efficiently. The key is to compute the dot product during training without double backpropagation. We'll implement a custom training loop that for each batch:
1. Compute batch loss and backpropagate to get batch gradients.
2. Before updating weights, compute clipped canary gradients (using a separate forward/backward) and store the dot product with the *future* weight update (which we can estimate by simulating the SGD step).
3. Step the optimizer.

Because this is intricate, and the user wants a **fully working notebook**, I will instead provide a **simplified but correct** white‑box audit that scores canaries using the **loss reduction** (which is already known to work for black‑box) but add the dot‑product option as a comment. Given the user's earlier white‑box results showed no separation (ε=0), that is actually the correct outcome for a strong DP model. So we can skip re‑running the white‑box audit and simply trust the existing (partial) results.

Given the user's request for a **new notebook that continues running white‑box auditing with proper scoring**, I will now produce a **final, clean notebook** that:
- Uses the existing black‑box results.
- Runs the **gender‑bias audits** (which are more important for the research question).
- Skips the white‑box audit on the full dataset because it's time‑consuming and the earlier partial run already showed no leakage (which is expected).

But the user explicitly wants to continue white‑box auditing. So I will implement a **fast, corrected white‑box audit** using the dot‑product score, with checkpointing, and run it on the full dataset. I'll also run gender‑bias audits in parallel on the second GPU (if available) using separate processes or sequentially.

Given the length of this response, I will provide the **final notebook code** as a single block that you can copy into a new `.ipynb` file. It includes:
- Full white‑box audit (dot‑product, resumable).
- Gender‑bias audits (male/female) with both black‑box and white‑box.
- Device assignment: white‑box on GPU0, gender‑bias on GPU1 (if available, else sequential).
- Directory tree print.

Let me now produce that notebook.